<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/phase9-portfolio-launch/03_conformity_report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 9: Conformity Report
**Goal**: Build a general-purpose Conformity Report generator
      covering EU AI Act (11 articles), NIST AI RMF (4
      functions), NIST AI 600-1 (4 generative AI risk
      requirements), and ISO/IEC 42001 (3 clauses), 22
      obligations total. Untested obligations are honestly
      marked NOT ASSESSED rather than omitted. Includes an
      adapter for real Phase 8 agent results, where one piece
      of evidence often satisfies obligations across multiple
      frameworks at once, and a manual intake path for
      obligations with no automated audit behind them.
**Regulatory mapping**: EU AI Act Articles 9, 10, 11, 12, 13,
                    14, 15, 18, 26, 27, 99; NIST AI RMF Govern,
                    Map, Measure, Manage; NIST AI 600-1
                    hallucination tracking, source data lineage,
                    output monitoring, confabulation prevention;
                    ISO/IEC 42001 Clauses 6, 8, 9.
**Date**: June 2026.
**Status**: In Progress.

In [1]:
import json
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any

# - EVIDENCE LEVELS (same discipline as the FRIA template) -
AUTOMATED = "AUTOMATED"
ATTESTED = "ATTESTED"
MISSING = "MISSING"

# - CONFORMITY STATUS -
COMPLIANT = "COMPLIANT"
NON_COMPLIANT = "NON_COMPLIANT"
PARTIAL = "PARTIAL"
NOT_ASSESSED = "NOT_ASSESSED"

# - THE LOCKED REGULATORY SCOPE, ACROSS THREE FRAMEWORKS -
# From the Afrispan Strategic Blueprint. EU AI Act gives 11 specific
# articles. NIST AI RMF gives 4 functions, not articles, since it is a
# risk-management methodology, not a list of legal obligations. ISO/IEC
# 42001 gives clauses, not articles, since it is a certifiable
# management-system standard. Each framework keeps its own native unit
# rather than being flattened into a single undifferentiated list, since
# a client needs to see which framework a given obligation belongs to.
REGULATORY_SCOPE = {
    "EU AI Act": {
        "Article 9": "Risk Management System",
        "Article 10": "Data Governance",
        "Article 11": "Technical Documentation",
        "Article 12": "Record-Keeping",
        "Article 13": "Transparency",
        "Article 14": "Human Oversight",
        "Article 15": "Accuracy, Robustness, Cybersecurity",
        "Article 18": "Quality Management System",
        "Article 26": "Deployer Obligations",
        "Article 27": "Fundamental Rights Impact Assessment",
        "Article 99": "Penalties",
    },
    "NIST AI RMF": {
        "GOVERN": "Organisational policies, culture, accountability for AI risk",
        "MAP": "Context and risk identification for the specific AI application",
        "MEASURE": "Quantitative and qualitative analysis of AI risks",
        "MANAGE": "Prioritising and acting on identified risks",
    },
    "NIST AI 600-1": {
        "Hallucination Tracking": "Detecting plausible but false model outputs",
        "Source Data Lineage": "Verifying outputs can be traced to source documents",
        "Output Monitoring": "Ongoing monitoring of generated content in production",
        "Confabulation Prevention": "Preventing fabricated sources and citations",
    },
    "ISO/IEC 42001": {
        "Clause 6": "Planning, AI risk assessment and treatment",
        "Clause 8": "Operation, AI system impact assessment and controls",
        "Clause 9": "Performance evaluation, monitoring and internal audit",
    },
}


@dataclass
class FrameworkConformity:
  """One obligation's conformity status, within one framework, with
  its evidence trail."""
  framework: str
  item: str
  obligation: str
  status: str = NOT_ASSESSED
  finding: str = ""
  evidence_level: str = MISSING
  source_reference: Optional[str] = None


@dataclass
class ConformityReportInput:
  """Generic Conformity Report input. Always covers the full locked
  scope across all three frameworks. Works for any audited system,
  any audit pipeline."""
  system_name: str
  system_purpose: str
  sector: str
  items: Dict[str, FrameworkConformity] = field(default_factory=dict)


def _scope_key(framework, item):
  return f"{framework} | {item}"


def _blank_scope() -> Dict[str, FrameworkConformity]:
  """Every report starts with every item, across every framework,
  present and NOT_ASSESSED, so nothing is silently missing."""
  items = {}
  for framework, obligations in REGULATORY_SCOPE.items():
    for item, obligation in obligations.items():
      key = _scope_key(framework, item)
      items[key] = FrameworkConformity(framework=framework, item=item, obligation=obligation)
  return items


# - ADAPTER: REAL PHASE 8 AGENT RESULTS TO CONFORMITY REPORT -
# One piece of Phase 8 evidence often satisfies obligations in more than
# one framework at once. The kill-switch and human authorization
# mechanism (Article 14) is also direct evidence for NIST's MANAGE
# function and, partially, ISO 42001's Clause 8 (operation, AI system
# controls). Each entry below lists every (framework, item) pair that
# a given agent's evidence speaks to, not just its EU AI Act article.
AGENT_SCOPE_MAP = {
    "Bias Audit Agent": [
        ("EU AI Act", "Article 10"),
        ("NIST AI RMF", "MEASURE"),
        ("ISO/IEC 42001", "Clause 6"),
    ],
    "Oversight Audit Agent": [
        ("EU AI Act", "Article 14"),
        ("NIST AI RMF", "MANAGE"),
        ("ISO/IEC 42001", "Clause 8"),
    ],
    "Documentation Audit Agent": [
        ("EU AI Act", "Article 11"),
        ("NIST AI RMF", "GOVERN"),
    ],
}

def from_agent_results(system, agent_results, audit_log_entries=None):
  """Build a ConformityReportInput from real Phase 8 agent_results.
  Optionally pass audit log entries so source_reference can point at a
  real hash instead of just the agent's own claim."""
  system_name = system.get("system_name") or system.get("name", "Unknown System")
  items = _blank_scope()

  log_by_agent = {}
  if audit_log_entries:
    for entry in audit_log_entries:
      if entry["event_type"] == "AGENT_COMPLETED":
        log_by_agent[entry["agent"]] = entry

  for result in agent_results:
    agent_name = result.get("agent")
    scope_pairs = AGENT_SCOPE_MAP.get(agent_name, [])
    if not scope_pairs:
      continue

    verdict = result.get("verdict")
    status = COMPLIANT if verdict == "PASS" else NON_COMPLIANT if verdict == "FAIL" else PARTIAL

    finding_text = result.get("findings") or result.get("finding") or ""
    if isinstance(finding_text, list):
      formatted_findings = []
      for f in finding_text:
        if isinstance(f, dict):
          dim = f.get("dimension", "unknown dimension")
          ratio = f.get("ratio")
          severity = f.get("severity", "")
          formatted_findings.append(f"{dim.replace('_', ' ')}: ratio {ratio} ({severity})")
        else:
          formatted_findings.append(str(f))
      finding_text = "; ".join(formatted_findings)

    log_entry = log_by_agent.get(agent_name)
    source_ref = (f"audit_log entry #{log_entry['entry_number']}, "
                  f"hash {log_entry['entry_hash'][:12]}...") if log_entry else None

    for framework, item in scope_pairs:
      key = _scope_key(framework, item)
      if key not in items:
        continue
      items[key] = FrameworkConformity(
          framework=framework,
          item=item,
          obligation=REGULATORY_SCOPE[framework][item],
          status=status,
          finding=f"{agent_name} verdict: {verdict}. {finding_text}".strip(),
          evidence_level=AUTOMATED if log_entry else ATTESTED,
          source_reference=source_ref,
      )

  return ConformityReportInput(
      system_name=system_name,
      system_purpose=system.get("purpose", "Not specified"),
      sector=system.get("sector", "Not specified"),
      items=items,
  )


# - MANUAL INTAKE: ONE ITEM AT A TIME -
def intake_item(report_input, framework, item, status, finding, source):
  """Update one (framework, item) conformity status from a manually
  collected answer. Anything not called this way stays NOT_ASSESSED."""
  key = _scope_key(framework, item)
  finding = (finding or "").strip()
  source = (source or "").strip()
  evidence_level = ATTESTED if (finding and source) else MISSING

  report_input.items[key] = FrameworkConformity(
      framework=framework,
      item=item,
      obligation=REGULATORY_SCOPE.get(framework, {}).get(item, "Unknown obligation"),
      status=status if (finding and source) else NOT_ASSESSED,
      finding=finding,
      evidence_level=evidence_level,
      source_reference=source if source else None,
  )
  return report_input


# - THE CONFORMITY REPORT GENERATOR -
def build_conformity_report(report_input):
  items = report_input.items

  status_counts = {COMPLIANT: 0, NON_COMPLIANT: 0, PARTIAL: 0, NOT_ASSESSED: 0}
  evidence_counts = {AUTOMATED: 0, ATTESTED: 0, MISSING: 0}
  for it in items.values():
    status_counts[it.status] += 1
    evidence_counts[it.evidence_level] += 1

  total = len(items)
  assessed = total - status_counts[NOT_ASSESSED]

  if status_counts[NON_COMPLIANT] > 0:
    overall = "NOT APPROVED, ONE OR MORE OBLIGATIONS NON-COMPLIANT"
  elif status_counts[PARTIAL] > 0:
    overall = "CONDITIONAL, ONE OR MORE OBLIGATIONS PARTIALLY COMPLIANT"
  elif assessed < total:
    overall = "INCOMPLETE ASSESSMENT, NOT ALL OBLIGATIONS ASSESSED"
  else:
    overall = "FULLY COMPLIANT"

  return {
      "system_name": report_input.system_name,
      "system_purpose": report_input.system_purpose,
      "sector": report_input.sector,
      "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
      "items": {k: asdict(v) for k, v in items.items()},
      "status_counts": status_counts,
      "evidence_counts": evidence_counts,
      "coverage_ratio": f"{assessed}/{total} obligations assessed",
      "automated_ratio": f"{evidence_counts[AUTOMATED]}/{total} obligations with automated evidence",
      "overall_verdict": overall,
  }


def render_conformity_markdown(report):
  status_badge = {
      COMPLIANT: "[COMPLIANT]",
      NON_COMPLIANT: "[NON-COMPLIANT]",
      PARTIAL: "[PARTIAL]",
      NOT_ASSESSED: "[NOT ASSESSED]",
  }
  evidence_badge = {
      AUTOMATED: "(automated evidence)",
      ATTESTED: "(manual attestation)",
      MISSING: "(no evidence)",
  }

  lines = [
      "# Conformity Report",
      f"**System:** {report['system_name']}",
      f"**Purpose:** {report['system_purpose']}",
      f"**Sector:** {report['sector']}",
      f"**Generated:** {report['generated_at']}",
      f"**Coverage:** {report['coverage_ratio']}",
      f"**Automated evidence:** {report['automated_ratio']}",
      "", "---", "",
  ]

  for framework, obligations in REGULATORY_SCOPE.items():
    lines.append(f"## {framework}")
    lines.append("")
    for item_key in obligations:
      key = _scope_key(framework, item_key)
      it = report["items"][key]
      badge = status_badge[it["status"]]
      ev_badge = evidence_badge[it["evidence_level"]]
      lines.append(f"### {it['item']}: {it['obligation']} {badge} {ev_badge}")
      if it["finding"]:
        lines.append(f"\n{it['finding']}")
      else:
        lines.append("\n*No finding recorded.*")
      if it["source_reference"]:
        lines.append(f"\n*Source: {it['source_reference']}*")
      lines.append("")
    lines.append("---")
    lines.append("")

  lines.append(f"**Overall verdict: {report['overall_verdict']}**")

  return "\n".join(lines)


# - DEMONSTRATION 1: FROM REAL PHASE 8 AGENT RESULTS -
print("====== DEMONSTRATION 1: CONFORMITY REPORT FROM REAL PHASE 8 RESULTS ======\n")

real_agent_results = [
    {"agent": "Bias Audit Agent", "article": "EU AI Act Article 10", "verdict": "FAIL",
     "critical_breach": True,
     "findings": [{"dimension": "gender_disparate_impact", "ratio": 0.43, "severity": "HIGH", "compliant": False},
                  {"dimension": "age_disparate_impact", "ratio": 0.20, "severity": "CRITICAL", "compliant": False},
                  {"dimension": "ethnicity_disparate_impact", "ratio": 0.80, "severity": "PASS", "compliant": True}]},
    {"agent": "Oversight Audit Agent", "article": "EU AI Act Article 14", "verdict": "FAIL",
     "critical_breach": True, "findings": "None currently implemented"},
    {"agent": "Documentation Audit Agent", "article": "EU AI Act Article 11 and 18", "verdict": "FAIL",
     "critical_breach": False, "finding": "Partial. No FRIA completed."},
]

real_audit_log = [
    {"entry_number": 3, "event_type": "AGENT_COMPLETED", "agent": "Bias Audit Agent",
     "entry_hash": "8bae181de688928f6e58542af28f81bf80d3306b4c0256ad072e9e116e51fc23"},
    {"entry_number": 7, "event_type": "AGENT_COMPLETED", "agent": "Oversight Audit Agent",
     "entry_hash": "5e6a5f7b40598c304f52e20cf37c2abe4f90ec10cf0a6cb95476895dcbd26f00"},
    {"entry_number": 11, "event_type": "AGENT_COMPLETED", "agent": "Documentation Audit Agent",
     "entry_hash": "326a167657733472a825b76464240671ece18073c8bd2bac405d30efd8918d97"},
]

real_system = {
    "system_name": "TalentMatch AI",
    "purpose": "Automated CV screening",
    "sector": "employment",
}

report_input = from_agent_results(real_system, real_agent_results, real_audit_log)
report = build_conformity_report(report_input)
print(render_conformity_markdown(report))


# - DEMONSTRATION 2: MANUAL INTAKE ADDS AN UNASSESSED OBLIGATION -
print("\n\n====== DEMONSTRATION 2: MANUAL INTAKE ADDS ARTICLE 27 ======\n")

report_input = intake_item(
    report_input, "EU AI Act", "Article 27",
    status=PARTIAL,
    finding="FRIA generator built and verified against real audit data, "
            "but not yet run for every high-risk use case in production.",
    source="Phase 9 FRIA Template deliverable, verified 2026-06-25",
)

report2 = build_conformity_report(report_input)
print(render_conformity_markdown(report2))


# - DEMONSTRATION 3: MANUAL INTAKE FILLS IN NIST AI 600-1 FROM REAL PHASE 7 FINDINGS -
# TalentMatch AI (Phase 8's audit subject) has no RAG component, so the
# three Phase 8 agents correctly leave NIST AI 600-1 untouched. The real
# evidence for AI 600-1 lives in Phase 7's RAG pipeline findings instead,
# a different system, added here via manual intake rather than invented
# or borrowed from an unrelated audit.
print("\n\n====== DEMONSTRATION 3: NIST AI 600-1 FROM REAL PHASE 7 FINDINGS ======\n")

report_input = intake_item(
    report_input, "NIST AI 600-1", "Hallucination Tracking",
    status=COMPLIANT,
    finding="Grounded prompt restricts answers to retrieved sources; "
            "4 of 4 test queries returned grounded responses with no "
            "ungrounded claims introduced.",
    source="Phase 7 Lesson 1 RAG Pipeline findings, June 2026",
)
report_input = intake_item(
    report_input, "NIST AI 600-1", "Source Data Lineage",
    status=COMPLIANT,
    finding="Every RAG response includes the exact source documents "
            "used to generate the answer, enabling verification against "
            "original text.",
    source="Phase 7 Lesson 1 RAG Pipeline findings, June 2026",
)
report_input = intake_item(
    report_input, "NIST AI 600-1", "Output Monitoring",
    status=COMPLIANT,
    finding="Retrieval scoring logged per query; semantic intent check "
            "adds a second monitoring layer ahead of generation.",
    source="Phase 7 Lesson 3 Semantic Intent Check findings, June 2026",
)
report_input = intake_item(
    report_input, "NIST AI 600-1", "Confabulation Prevention",
    status=PARTIAL,
    finding="Model instructed to state when source documents do not "
            "contain the answer. Retrieval quality limitation found in "
            "testing (Article 5 vs Article 99 confusion under keyword "
            "scoring), since resolved by the Phase 9 Chroma migration.",
    source="Phase 7 Lesson 1 Finding 2, resolved per Phase 9 Section 1",
)

report3 = build_conformity_report(report_input)
print(render_conformity_markdown(report3))

print("\n\n====== COMPARISON ======")
print(f"Demo 1: {report['coverage_ratio']}, {report['overall_verdict']}")
print(f"Demo 2: {report2['coverage_ratio']}, {report2['overall_verdict']}")
print(f"Demo 3: {report3['coverage_ratio']}, {report3['overall_verdict']}")
print(f"\nTotal obligations in scope across all 4 frameworks: {len(report_input.items)}")

====== DEMONSTRATION 1: CONFORMITY REPORT FROM REAL PHASE 8 RESULTS ======

# Conformity Report
**System:** TalentMatch AI
**Purpose:** Automated CV screening
**Sector:** employment
**Generated:** 2026-06-25 16:20:28
**Coverage:** 8/22 obligations assessed
**Automated evidence:** 8/22 obligations with automated evidence

---

## EU AI Act

### Article 9: Risk Management System [NOT ASSESSED] (no evidence)

*No finding recorded.*

### Article 10: Data Governance [NON-COMPLIANT] (automated evidence)

Bias Audit Agent verdict: FAIL. gender disparate impact: ratio 0.43 (HIGH); age disparate impact: ratio 0.2 (CRITICAL); ethnicity disparate impact: ratio 0.8 (PASS)

*Source: audit_log entry #3, hash 8bae181de688...*

### Article 11: Technical Documentation [NON-COMPLIANT] (automated evidence)

Documentation Audit Agent verdict: FAIL. Partial. No FRIA completed.

*Source: audit_log entry #11, hash 326a16765773...*

### Article 12: Record-Keeping [NOT ASSESSED] (no evidence)

*No finding reco

## Findings: Conformity Report

**Frameworks covered:** EU AI Act (11 articles), NIST AI RMF
                     (4 functions), NIST AI 600-1 (4 generative
                     AI risk requirements), ISO/IEC 42001
                     (3 clauses), 22 obligations total
**Demonstrations:** 3 (Phase 8 automated results, manual intake
                     adding Article 27, manual intake adding
                     NIST AI 600-1 from real Phase 7 evidence)
**Date:** June 2026
**Regulatory mapping:** EU AI Act, NIST AI RMF, NIST AI 600-1,
                     ISO/IEC 42001

### What Was Built

A Conformity Report generator covering four frameworks on every
report, not just the EU AI Act. Untested obligations are marked
NOT ASSESSED rather than silently omitted, so a report's
declared scope is always the full 22 obligations regardless of
how much evidence currently backs it.

The from_agent_results() adapter maps one piece of Phase 8
evidence to every framework it actually satisfies at once. The
Bias Audit Agent's finding, for example, populates EU AI Act
Article 10, NIST's MEASURE function, and ISO Clause 6
simultaneously, since one disparate-impact test is genuinely
risk measurement under any framework's vocabulary.

### An Omission Found and Corrected

The first version of this generator covered EU AI Act, NIST AI
RMF, and ISO/IEC 42001, but left out NIST AI 600-1 (the
Generative AI Profile), despite it being established, proven
work from Phase 7 (hallucination tracking, source data lineage,
output monitoring, confabulation prevention, all already mapped
in the Phase 7 Lesson 1 findings). This was a real gap in
research, not a deliberate scoping choice, and it mattered
specifically because every system in this portfolio involves an
LLM, putting AI 600-1 squarely in scope. The gap was corrected
by adding it as its own framework entry.

### Key Findings

1. TalentMatch AI (Phase 8's audit subject) correctly shows all
   four NIST AI 600-1 items as NOT ASSESSED in Demonstrations 1
   and 2, since it has no RAG component and was never tested
   against generative-AI-specific risks. The generator did not
   fabricate a connection that does not exist.

2. Demonstration 3 fills in NIST AI 600-1 using real Phase 7
   findings, a genuinely different system's evidence, added
   honestly via manual intake rather than borrowed silently
   from an unrelated audit.

3. The Confabulation Prevention finding traces a complete
   audit-to-remediation lifecycle in a single line: the Article
   5 vs Article 99 retrieval bug found in Phase 7 testing, and
   its actual fix via the Phase 9 Chroma migration. This is
   exactly the kind of longitudinal evidence the "AI Governance
   in Practice" document should draw on directly.

4. Coverage climbed from 8/22 to 9/22 to 13/22 across the three
   demonstrations, while the overall verdict stayed correctly
   NOT APPROVED throughout, since TalentMatch AI's real Article
   10 and Article 14 failures were never actually resolved.
   Adding more evidence elsewhere does not launder an unresolved
   critical finding.

### Recommendation
Proceed to the cross-border framework deliverable. The 22-item
scope built here is the natural foundation: cross-border work
extends the same structure to jurisdictions beyond the EU,
rather than starting a new scope from zero.